# NIH Concept Activation Check

Use this notebook to sanity-check whether your CLIP prompts activate on NIH ChestXray14 images before training the CBM.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from clip_loader import load_clip_encoder
import data_utils

torch.set_grad_enabled(False)
plt.style.use('seaborn-v0_8')

In [ ]:
# --- Configuration ---
NIH_IMG_DIR = os.environ.get('NIH_CXR_IMG_DIR', '/PATH/TO/NIH/images-224')
CONCEPT_PATH = 'data/concept_sets/nih14_filtered.txt'
DATASET_SPLIT = 'nih14_train'   # or 'nih14_val'
CLIP_MODEL = 'ViT-B/16'
BATCH_SIZE = 64
MAX_SAMPLES = 1024   # set to None for all images
TOPK = 5
HIT_THRESHOLD = 0.3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

assert os.path.isdir(NIH_IMG_DIR), f'NIH images not found at {NIH_IMG_DIR}'
assert os.path.isfile(CONCEPT_PATH), f'Missing concept file: {CONCEPT_PATH}'
print(f'Using device: {DEVICE}')

In [ ]:
# --- Dataset & Model ---
data_utils.configure_nih_dataset(img_dir=NIH_IMG_DIR)
clip_wrapper = load_clip_encoder(CLIP_MODEL, DEVICE)
clip_model = clip_wrapper
clip_preprocess = clip_wrapper.preprocess
dataset = data_utils.get_data(DATASET_SPLIT, preprocess=clip_preprocess)
indices = list(range(len(dataset)))
random.shuffle(indices)
if MAX_SAMPLES is not None:
    indices = indices[:MAX_SAMPLES]
subset = Subset(dataset, indices)
loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=4, pin_memory=True)
print(f'Evaluating {len(indices)} images from {DATASET_SPLIT}')

with open(CONCEPT_PATH) as f:
    raw_concepts = [line.strip() for line in f if line.strip()]
text_prompts = [f'a chest radiograph showing {concept}' for concept in raw_concepts]
text_tokens = clip_wrapper.tokenize(text_prompts).to(DEVICE)
text_features = clip_model.encode_text(text_tokens)
text_features = text_features / text_features.norm(dim=-1, keepdim=True)
print(f'Loaded {len(raw_concepts)} concepts')

In [ ]:
# --- Similarity Computation ---
all_scores = []
topk_records = []

for images, _ in tqdm(loader, desc='Computing similarities'):
    images = images.to(DEVICE)
    feats = clip_model.encode_image(images)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    sims = feats @ text_features.T
    all_scores.append(sims.cpu())
    top_vals, top_idx = sims.topk(TOPK, dim=-1)
    for vals, idxs in zip(top_vals.cpu(), top_idx.cpu()):
        topk_records.append({
            'top_concepts': [raw_concepts[i] for i in idxs.tolist()],
            'scores': [float(v) for v in vals]
        })

all_scores = torch.cat(all_scores, dim=0)
print('Similarity matrix shape:', all_scores.shape)


In [ ]:
# --- Aggregate Metrics ---
max_per_concept = all_scores.max(dim=0).values
hit_rate = (all_scores > HIT_THRESHOLD).float().mean(dim=0)
mean_score = all_scores.mean(dim=0)
summary_df = pd.DataFrame({
    'concept': raw_concepts,
    'max_similarity': max_per_concept.numpy(),
    'mean_similarity': mean_score.numpy(),
    'hit_rate': hit_rate.numpy()
}).sort_values('max_similarity', ascending=False).reset_index(drop=True)
summary_df.head(20)


In [ ]:
# --- Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(max_per_concept.numpy(), bins=30, color='steelblue')
axes[0].set_title('Max similarity per concept')
axes[0].set_xlabel('Similarity')
axes[0].set_ylabel('Count')

axes[1].hist(hit_rate.numpy(), bins=30, color='salmon')
axes[1].set_title(f'Hit rate (> {HIT_THRESHOLD}) per concept')
axes[1].set_xlabel('Fraction of images above threshold')
plt.tight_layout()
plt.show()


In [ ]:
# Inspect a few sample predictions
pd.DataFrame(topk_records[:10])
